# Getting started: check your setup (R)

**Goal.** Confirm that this notebook runs in the Trees from Space R environment, that it can write files, whether this machine can reach NASA's data servers, and whether your Earthdata credentials work. The notebook saves a short report of what it found.

**Time.** About 5 minutes. **Data downloaded.** None (one small metadata request when online).

**Before starting.** Create the R environment and register its kernel ([environment guide](../../docs/getting-started/environments.md)), and store your Earthdata credentials in `~/.netrc` ([Earthdata Login guide](../../docs/getting-started/earthdata-login.md)). On CHPC, follow the [CHPC guide](../../docs/getting-started/chpc.md). If you are new to notebooks, read [Notebook basics](../../docs/getting-started/notebook-basics.md). Choose the **R 4.5 (Trees from Space)** kernel.

Run the cells in order with **Shift + Enter**. Each check prints `OK` or explains what to fix.

## Settings

Everything you might change is in the next cell. You can leave these values as they are.

In [ ]:
# Open one small NASA data file to test the full download route (needs internet access).
CHECK_DATA_ACCESS <- TRUE

# Seconds to wait when testing whether this machine can reach NASA.
NETWORK_TIMEOUT_S <- 5

## 1. The environment

The next cell loads the packages the R lessons use and prints their versions.

In [ ]:
required <- c("terra", "sf", "rstac", "jsonlite")
missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
versions <- list(R = paste(R.version$major, R.version$minor, sep = "."))
for (pkg in setdiff(required, missing)) versions[[pkg]] <- as.character(packageVersion(pkg))

cat("R", versions$R, "at", R.home(), "\n")
for (pkg in names(versions)[-1]) cat(sprintf("  %-10s %s\n", pkg, versions[[pkg]]))
if (length(missing) > 0) {
  cat("\nMissing packages:", paste(missing, collapse = ", "), "\n")
  cat("Choose the 'R 4.5 (Trees from Space)' kernel, or create the environment first.\n")
} else {
  suppressPackageStartupMessages(library(terra))
  versions$gdal <- gdal()
  cat(sprintf("  %-10s %s\n", "gdal", versions$gdal))
  cat("\nOK: all lesson packages are available.\n")
}

## 2. Where this notebook runs and saves files

Lessons save downloaded data to a `cache/` folder and results to an `outputs/` folder at the top of the repository. You can move them by setting the `TFS_CACHE_DIR` and `TFS_OUTPUT_DIR` environment variables ([environment guide](../../docs/getting-started/environments.md)). The next cell finds these folders and checks that it can write to them.

In [ ]:
find_repo_root <- function(start = getwd()) {
  # Walk up from `start` to the folder holding the .tfs-root marker file.
  folder <- normalizePath(start)
  repeat {
    if (file.exists(file.path(folder, ".tfs-root"))) return(folder)
    parent <- dirname(folder)
    if (parent == folder) return(NULL)
    folder <- parent
  }
}

detect_platform <- function() {
  # Name the kind of machine running this notebook (no hostnames are saved).
  if (nzchar(Sys.getenv("COLAB_RELEASE_TAG"))) return("colab")
  host <- Sys.info()[["nodename"]]
  on_chpc <- grepl("chpc.utah.edu", host, fixed = TRUE) || dir.exists("/scratch/general/vast")
  if (!on_chpc) return("local")
  if (startsWith(host, "frisco")) return("chpc-frisco")
  partition <- Sys.getenv("SLURM_JOB_PARTITION")
  if (nzchar(partition)) paste0("chpc-", partition) else "chpc-login"
}

env_or <- function(name, default) {
  value <- Sys.getenv(name)
  if (nzchar(value)) value else default
}

PLATFORM <- detect_platform()
REPO_ROOT <- find_repo_root()
BASE_DIR <- if (is.null(REPO_ROOT)) getwd() else REPO_ROOT
CACHE_DIR <- env_or("TFS_CACHE_DIR", file.path(BASE_DIR, "cache"))
OUTPUT_DIR <- env_or("TFS_OUTPUT_DIR", file.path(BASE_DIR, "outputs"))

cat("Platform:", PLATFORM, "\n")
if (is.null(REPO_ROOT)) cat("Note: this notebook is not inside the trees-from-space folder; using the current folder.\n")

paths_writable <- TRUE
for (folder in c(CACHE_DIR, OUTPUT_DIR)) {
  dir.create(folder, recursive = TRUE, showWarnings = FALSE)
  probe <- file.path(folder, ".write-test")
  ok <- tryCatch({ writeLines("ok", probe); file.remove(probe) }, error = function(e) FALSE,
                 warning = function(w) FALSE)
  if (isTRUE(ok)) {
    cat("OK: can write to", folder, "\n")
  } else {
    paths_writable <- FALSE
    cat("Cannot write to", folder, "\n")
  }
}

## 3. Can this machine reach NASA?

NASA data live on servers on the internet. Not every machine can reach them: some CHPC compute nodes have no internet access, although OnDemand sessions on the default interactive partition could reach NASA in our tests. The next cell tries to contact NASA's search service and stops waiting after a few seconds.

In [ ]:
network_available <- function(host = "cmr.earthdata.nasa.gov", timeout_s = 5) {
  # Return TRUE if an HTTPS request to `host` succeeds.
  if (Sys.getenv("TFS_OFFLINE") == "1") return(FALSE)  # lets you test the offline route
  handle <- curl::new_handle(connecttimeout = timeout_s, timeout = 3 * timeout_s, nobody = TRUE)
  tryCatch({
    curl::curl_fetch_memory(paste0("https://", host, "/search/"), handle = handle)
    TRUE
  }, error = function(e) FALSE)
}

ONLINE <- network_available(timeout_s = NETWORK_TIMEOUT_S)
if (ONLINE) {
  cat("OK: this machine can reach NASA. Lessons can search for and download data here.\n")
} else if (startsWith(PLATFORM, "chpc")) {
  cat("This CHPC machine cannot reach the internet. Some compute nodes have no internet access.\n")
  cat("Lessons that download data need a first run on a machine with internet access:\n")
  cat("see 'Running a lesson that downloads data' in docs/getting-started/chpc.md.\n")
} else {
  cat("No connection to NASA. Check your internet connection, then rerun this cell.\n")
}

## 4. Earthdata credentials

The lessons read your Earthdata username and password from the file `~/.netrc` in your home directory. The next cell checks that the file exists, that only you can read it, and that it has an entry for NASA Earthdata. It never prints the credentials.

The R lessons point GDAL, the library that reads the data files, straight at `~/.netrc`. They do not use `earthdatalogin::edl_netrc()`, which writes a shared public account when it finds no credentials in its own location.

In [ ]:
NETRC_PATH <- env_or("NETRC", path.expand("~/.netrc"))
EARTHDATA_HOST <- "urs.earthdata.nasa.gov"

netrc_status <- function(path) {
  # Report whether `path` exists, is private, and holds an Earthdata entry.
  status <- list(present = file.exists(path), private = FALSE, has_earthdata = FALSE)
  if (status$present) {
    mode <- as.integer(file.info(path)$mode)
    status$private <- bitwAnd(mode, strtoi("77", 8L)) == 0
    status$has_earthdata <- any(grepl(EARTHDATA_HOST, readLines(path, warn = FALSE), fixed = TRUE))
  }
  status
}

NETRC <- netrc_status(NETRC_PATH)
if (!NETRC$present) {
  cat("No credentials file at", NETRC_PATH, "- see docs/getting-started/earthdata-login.md.\n")
} else if (!NETRC$has_earthdata) {
  cat(NETRC_PATH, "has no entry for", EARTHDATA_HOST, "- see docs/getting-started/earthdata-login.md.\n")
} else {
  cat("OK:", NETRC_PATH, "has an Earthdata entry.\n")
}
if (NETRC$present && !NETRC$private) {
  cat("Other users can read", NETRC_PATH, "- fix this with:  chmod 600 ~/.netrc\n")
}

## 5. Open one NASA file

When this machine is online, the next cell opens the header of one Harmonized Landsat Sentinel-2 (HLS) quality file without downloading the whole file. This is the same route the R lessons use to read data, through GDAL and your `~/.netrc`, so it also confirms that NASA accepts your credentials.

In [ ]:
TEST_URL <- paste0(
  "https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSS30.020/",
  "HLS.S30.T12SVH.2024167T180919.v2.0/HLS.S30.T12SVH.2024167T180919.v2.0.Fmask.tif"
)
cookie_dir <- path.expand("~/.cache/trees-from-space")
dir.create(cookie_dir, recursive = TRUE, showWarnings = FALSE)
Sys.chmod(cookie_dir, mode = "0700")
cookie_file <- file.path(cookie_dir, "earthdata-cookies.txt")
Sys.setenv(
  GDAL_HTTP_NETRC = "YES",
  GDAL_HTTP_NETRC_FILE = NETRC_PATH,
  GDAL_HTTP_COOKIEFILE = cookie_file,
  GDAL_HTTP_COOKIEJAR = cookie_file,
  GDAL_DISABLE_READDIR_ON_OPEN = "EMPTY_DIR",
  CPL_VSIL_CURL_ALLOWED_EXTENSIONS = ".tif",
  GDAL_HTTP_MAX_RETRY = "3",
  GDAL_HTTP_RETRY_DELAY = "2",
  GDAL_HTTP_CONNECTTIMEOUT = "10",
  GDAL_HTTP_TIMEOUT = "60"
)

access_result <- "skipped (offline)"
if (ONLINE && NETRC$has_earthdata && CHECK_DATA_ACCESS) {
  access_result <- tryCatch({
    test_file <- terra::rast(paste0("/vsicurl/", TEST_URL))
    cat("Opened", basename(TEST_URL), ":", ncol(test_file), "x", nrow(test_file), "pixels,",
        paste0("EPSG:", terra::crs(test_file, describe = TRUE)$code), "\n")
    "ok"
  }, error = function(e) {
    cat("Could not open the test file:", conditionMessage(e), "\n")
    "failed"
  })
  cat("Data access:", access_result, "\n")
} else if (ONLINE) {
  access_result <- "skipped (no credentials)"
  cat("Skipped: add your Earthdata credentials first (section 4).\n")
} else {
  cat("Skipped: this machine is offline (section 3).\n")
}

## 6. Save the report

The last cell saves what this notebook found to `outputs/getting-started/` and prints it. The report contains no usernames, passwords, hostnames, or personal paths, so you can share it when asking for help.

In [ ]:
report <- list(
  notebook = "getting-started-r",
  run_utc = format(Sys.time(), "%Y-%m-%dT%H:%M:%S+00:00", tz = "UTC"),
  platform = PLATFORM,
  packages = versions,
  missing_packages = missing,
  paths_writable = paths_writable,
  network = ONLINE,
  netrc = NETRC,
  data_access = access_result
)
report_path <- file.path(OUTPUT_DIR, "getting-started", "environment-report-r.json")
dir.create(dirname(report_path), recursive = TRUE, showWarnings = FALSE)
jsonlite::write_json(report, report_path, auto_unbox = TRUE, pretty = TRUE)
cat("Saved", report_path, "\n\n")
cat(jsonlite::toJSON(report, auto_unbox = TRUE, pretty = TRUE), "\n")

## What next

- **Everything OK and online:** open a lesson from the [tutorial index](../README.md) and run it here.
- **Offline on CHPC:** some compute nodes have no internet access. Lessons that download data then need a short transfer job first, as described in the [CHPC guide](../../docs/getting-started/chpc.md).
- **Something failed:** the message above each result says what to fix. Rerun the notebook afterwards with *Restart Kernel and Run All Cells*.